Load and Use .h5 File in Colab

In [11]:
from google.colab import files
uploaded = files.upload()

Saving lstm_meta_stock_model.h5 to lstm_meta_stock_model (1).h5



Install TensorFlow (if not already)


In [12]:
import tensorflow as tf

Load the Model

In [13]:
model = tf.keras.models.load_model('/content/lstm_meta_stock_model.h5')


Use the Model

In [8]:
import numpy as np
import tensorflow as tf

# Load the model (already present in your code)
model = tf.keras.models.load_model('/content/lstm_meta_stock_model.h5')

# --- Add this section to inspect the expected input shape ---
expected_features = None
expected_timesteps = 10 # Default timestep value, adjust if your model requires a different fixed timestep

try:
    # Try to get the input shape from the model's input layer
    if hasattr(model, 'input_shape') and isinstance(model.input_shape, tuple):
        # Model input shape for sequential or functional API might be (None, timesteps, features)
        if len(model.input_shape) > 1:
            expected_timesteps = model.input_shape[-2] if model.input_shape[-2] is not None else expected_timesteps
            expected_features = model.input_shape[-1]
            print(f"Determined expected input shape from model.input_shape: {model.input_shape}")
        elif len(model.input_shape) == 1:
             # Might be a single dimension input, which is not typical for LSTM
             print(f"Model has a simple input_shape: {model.input_shape}. Cannot determine time series features directly.")

    # If not found directly on the model, try the first layer again
    if expected_features is None and len(model.layers) > 0:
         first_layer_input_shape = model.layers[0].input_shape
         if isinstance(first_layer_input_shape, tuple):
             # The first layer's input shape might be a list if the model has multiple inputs,
             # or a tuple for a single input layer.
             if isinstance(first_layer_input_shape, list):
                 # If it's a list, take the shape of the first input
                 if len(first_layer_input_shape) > 0 and isinstance(first_layer_input_shape[0], tuple):
                     if len(first_layer_input_shape[0]) > 1:
                          expected_timesteps = first_layer_input_shape[0][-2] if first_layer_input_shape[0][-2] is not None else expected_timesteps
                          expected_features = first_layer_input_shape[0][-1]
                          print(f"Determined expected input shape from first layer's input_shape (list): {first_layer_input_shape[0]}")
             elif len(first_layer_input_shape) > 1:
                 # Handle cases like (None, timesteps, features) or (timesteps, features)
                 expected_timesteps = first_layer_input_shape[-2] if first_layer_input_shape[-2] is not None else expected_timesteps
                 expected_features = first_layer_input_shape[-1]
                 print(f"Determined expected input shape from first layer's input_shape (tuple): {first_layer_input_shape}")
             else:
                  print(f"First layer's input_shape is {first_layer_input_shape}. Cannot determine time series features directly.")
         else:
              print(f"First layer's input_shape is not a tuple: {first_layer_input_shape}. Cannot determine features.")


except Exception as e:
    print(f"An error occurred while trying to determine input shape: {e}")
    print("Falling back to default dummy data configuration.")

# Use the determined features, or default if determination failed
if expected_features is not None:
    print(f"Model expects input with {expected_features} features per timestep and {expected_timesteps} timesteps.")
    # Create dummy X_test with the correct number of features and timesteps
    # Keep the batch size as 1
    X_test = np.random.rand(1, expected_timesteps, expected_features).astype(np.float32)
    X_test = tf.convert_to_tensor(X_test)
    print(f"Generated dummy X_test with shape: {X_test.shape}")
else:
    print("Could not reliably determine expected features from the model.")
    print("Please inspect your model to find the expected input shape.")
    # As a last resort, you might try printing model.summary() to see the layer shapes
    # model.summary() # Uncomment this line if you want to see the summary
    # Or you might know the expected features from your training data
    # expected_features = <manually enter correct number of features here>
    # X_test = np.random.rand(1, expected_timesteps, expected_features).astype(np.float32)
    # X_test = tf.convert_to_tensor(X_test)
    # print(f"Using manually set dummy X_test shape: {X_test.shape}")

    # If you cannot determine the shape, you cannot proceed with prediction
    raise ValueError("Could not determine the expected number of features for the model input.")


# Example prediction (assuming X_test is preprocessed)
predictions = model.predict(X_test)

print(predictions)

Determined expected input shape from model.input_shape: (None, 60, 1)
Model expects input with 1 features per timestep and 60 timesteps.
Generated dummy X_test with shape: (1, 60, 1)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step
[[0.4991672]]


Check Model Summary

In [14]:
model.summary()


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 60, 50)         │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 60, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,653 (119.74 KB)

 Trainable params: 30,651 (119.73 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)